# Full evaluation notebook (Kaggle T4 x2) -- A6 + A6.5

Runs the *full* (unsampled) benchmarks against both `d4` and `d6` SFT checkpoints, on GPU,
instead of the small local CPU spot-checks:

- `scripts/chat_eval.py`: ARC-Easy, ARC-Challenge, MMLU, GSM8K, HumanEval -- upstream nanochat's
  own benchmark suite. Expected near-baseline at this scale (see README/RESEARCH_LOG), run anyway
  for completeness.
- `scripts/eval_blimp.py`: BLiMP (Warstadt et al. 2020), 67 grammar categories x 1000 minimal
  pairs each -- a much better fit for what a model this size can plausibly do well on. Local
  spot-checks (3 categories x 30 pairs) already showed 73-93%, well above the 50% chance level.

Only GPU 0 is used for nothing here -- these are single-process, no `torchrun`/DDP needed (unlike
training), so this notebook doesn't split work across both T4s. That's fine; the point of T4 x2
here is just "whichever GPU accelerator is available", not parallelism.

Rough time budget (estimate, not measured yet): full `chat_eval.py` for one model is dominated by
the generative tasks (GSM8K ~1319 problems, HumanEval ~164, evaluated one at a time) -- maybe
45-70 min per model. Full `eval_blimp.py` (67 x 1000 pairs, batched) -- maybe 20-45 min per model.
Both models: **rough total 3-4 hours**. Cut `--max-problems`/`--max-pairs` down if that's too much
for the remaining GPU-hours budget -- see the commented-out fast alternative in each cell.

Upload via File -> Upload Notebook. Same 4 Kaggle Secrets as the training notebooks, T4 x2
accelerator, internet access (also needs to download ARC/MMLU/GSM8K/HumanEval/BLiMP from the HF
Hub on first run).

## Cell 1: clone repo, install dependencies

In [ ]:
import os
import subprocess
import sys

REPO_URL = "https://github.com/nadeko0/nanochat-ru.git"
REPO_DIR = "/kaggle/working/repo"

if os.path.isdir(os.path.join(REPO_DIR, ".git")):
    print("Repo already present, pulling latest...")
    !git -C {REPO_DIR} pull
else:
    !git clone {REPO_URL} {REPO_DIR}

os.chdir(REPO_DIR)

def have(cmd):
    return subprocess.run(["bash", "-lc", f"command -v {cmd}"], capture_output=True).returncode == 0

if not have("uv"):
    !curl -LsSf https://astral.sh/uv/install.sh | sh
os.environ["PATH"] = f"{os.path.expanduser('~/.local/bin')}:{os.environ['PATH']}"

if not have("cargo"):
    !curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y
os.environ["PATH"] = f"{os.path.expanduser('~/.cargo/bin')}:{os.environ['PATH']}"

if not have("rclone"):
    !curl https://rclone.org/install.sh | sudo bash

!uv pip install --system --python {sys.executable} --extra gpu -r pyproject.toml

print("Cell 1 done.")

## Cell 2: configure rclone, pull both d4 and d6 SFT checkpoints + tokenizer

In [ ]:
import os
import subprocess
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
client_id = secrets.get_secret("GDRIVE_CLIENT_ID").strip()
client_secret = secrets.get_secret("GDRIVE_CLIENT_SECRET").strip()
oauth_token = secrets.get_secret("GDRIVE_OAUTH_TOKEN").strip()
folder_id = secrets.get_secret("GDRIVE_FOLDER_ID").strip()

rclone_conf_dir = os.path.expanduser("~/.config/rclone")
os.makedirs(rclone_conf_dir, exist_ok=True)
with open(os.path.join(rclone_conf_dir, "rclone.conf"), "w") as f:
    f.write(
        "[gdrive]\n"
        "type = drive\n"
        "scope = drive\n"
        f"client_id = {client_id}\n"
        f"client_secret = {client_secret}\n"
        f"token = {oauth_token}\n"
        f"root_folder_id = {folder_id}\n"
        "team_drive =\n"
    )

!rclone lsd gdrive:

DRIVE_REMOTE = "gdrive:"
NANOCHAT_BASE_DIR = "/kaggle/working/nanochat_cache"
os.environ["NANOCHAT_BASE_DIR"] = NANOCHAT_BASE_DIR
os.makedirs(NANOCHAT_BASE_DIR, exist_ok=True)

!rclone copy gdrive:tokenizer {NANOCHAT_BASE_DIR}/tokenizer --checksum -v
for tag in ["d4", "d6"]:
    !rclone copy gdrive:chatsft_checkpoints/{tag} {NANOCHAT_BASE_DIR}/chatsft_checkpoints/{tag} --checksum -v

print("Checkpoints ready:")
!ls {NANOCHAT_BASE_DIR}/chatsft_checkpoints/d4 {NANOCHAT_BASE_DIR}/chatsft_checkpoints/d6

## Cell 3: full chat_eval.py -- d4

In [ ]:
import os
os.chdir("/kaggle/working/repo")

# Full run, no -x limit. If this is taking too long / too much GPU budget, interrupt and rerun
# with e.g. `-x 200` (chat_eval.py's --max-problems flag) for a faster, still-informative sample.
!python -m scripts.chat_eval -i sft -g d4 2>&1 | tee /kaggle/working/chat_eval_d4.log

## Cell 4: full chat_eval.py -- d6

In [ ]:
import os
os.chdir("/kaggle/working/repo")

!python -m scripts.chat_eval -i sft -g d6 2>&1 | tee /kaggle/working/chat_eval_d6.log

## Cell 5: full BLiMP eval -- d4

In [ ]:
import os
os.chdir("/kaggle/working/repo")

# All 67 categories, 1000 pairs each, batched. If this is too slow, lower --max-pairs (e.g. 200).
!python -m scripts.eval_blimp -i sft -g d4 --batch-size 64 2>&1 | tee /kaggle/working/blimp_d4.log

## Cell 6: full BLiMP eval -- d6

In [ ]:
import os
os.chdir("/kaggle/working/repo")

!python -m scripts.eval_blimp -i sft -g d6 --batch-size 64 2>&1 | tee /kaggle/working/blimp_d6.log